# Open Door Legal — Client Feedback Analysis

Comparative analysis of 750 client feedback responses across two periods:

- **prior** — Mar 2024 – May 2025 (395 responses)
- **focus** — Jun 2025 – Aug 2026 (355 responses)

Sections:
1. Setup & load
2. Text cleanup (mojibake repair)
3. Quantitative overview
4. Statistical comparison: focus vs prior
5. Translation
6. Topic modeling — what ODL does well
7. Topic modeling — what could be better
8. Topic modeling — combined issue comments
9. Export & push to GitHub

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn nltk openpyxl deep-translator statsmodels -q
import nltk
nltk.download('stopwords', quiet=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from deep_translator import GoogleTranslator

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

RNG = np.random.default_rng(42)
N_BOOT = 2000          # bootstrap resamples for effect-size CIs

STOP_WORDS = set(stopwords.words('english'))
ODL_STOPS  = {'open', 'door', 'legal', 'odl', 'help', 'helped',
              'would', 'could', 'also', 'one', 'really', 'think',
              'feel', 'know', 'get', 'got', 'make', 'made', 'like',
              'need', 'needed', 'nothing', 'everything', 'satisfied'}
ALL_STOPS  = STOP_WORDS | ODL_STOPS

# Slide-deck palette, taken from the deck's theme part. Defined here in setup so any
# section can render a themed figure; see make_themed_figures.py for the full rationale.
SLIDE = {
    'surface': '#000026',   # deck background (theme lt1)
    'ink':     '#E7E7E7',   # deck body text  (theme dk1)
    'muted':   '#A9B2CE',
    'grid':    '#1B1F4A',
    'focus':   '#4A85E8',
    'prior':   '#21A896',
}

## 1. Load Data

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/feedback_comparative_periods.xlsx')

df = df.rename(columns={
    'Client Feedback: Created Date':           'date',
    'Case: Client Name':                       'client_name',
    'Case: Case Number':                       'case_number',
    'Case: Subject':                           'case_subject',
    'Case: Degree of Resolution':              'resolution',
    'Case: Case Owner':                        'case_owner',
    'Net Promoter Score':                      'nps',
    'How much of a positive diff has ODL had': 'positive_diff',
    'Program staff helped me meet my needs':   'needs_met_current',
    'What does Open Door Legal do well':       'do_well',
    'What could Open Door Legal do better':    'do_better',
    'Informed About Case':                     'informed',
    'Did we give choices':                     'gave_choices',
    'Barriers to services':                    'barriers',
    'Anything we could not have helped with?': 'beyond_scope',
    'Additional Comments':                     'additional_comments',
    'Barriers to services comment':            'barriers_comment',
    'Anything we could have helped comment':   'beyond_scope_comment',
    'How well has ODL met your needs':         'needs_met_legacy',
    'Needs Met (agreement scale)':             'needs_met_agreement',
    'Needs Met (numeric)':                     'needs_met_num',
    'Needs Met (source)':                      'needs_met_source',
    'Positive Diff (numeric)':                 'positive_diff_num',
    'NPS Category':                            'nps_category',
    'NPS Promoter (1/0)':                      'nps_promoter',
    'Gave Choices (1=Yes)':                    'gave_choices_bin',
    'Barriers Reported (1=Yes)':               'barriers_bin',
    'Unmet Need (1=Yes)':                      'unmet_need_bin',
    'Resolution Positive (1/0)':               'resolution_positive',
})

df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Short flag for which Needs Met wording each response saw
df['needs_scale'] = np.where(
    df['needs_met_source'].astype(str).str.startswith('current'), 'current', 'legacy')

# Rank the positive-difference categories by their own numeric codes rather than by a
# hardcoded list, so relabelled or added categories are picked up instead of dropped.
PD_ORDER = (df.dropna(subset=['positive_diff', 'positive_diff_num'])
              .groupby('positive_diff')['positive_diff_num'].first()
              .sort_values().index.tolist())
_ramp = ['#e74c3c', '#e59866', '#f4d03f', '#82e0aa', '#2ecc71']
PD_COLORS = {c: _ramp[round(i * (len(_ramp) - 1) / max(len(PD_ORDER) - 1, 1))]
             for i, c in enumerate(PD_ORDER)}

print(f'Positive-difference scale (worst to best): {" < ".join(PD_ORDER)}')
print()

print(f'Loaded {len(df):,} responses')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print()
print(df.groupby('period')['date'].agg(['min', 'max', 'count']))
print()
print('Needs Met scale by period:')
print(pd.crosstab(df['needs_scale'], df['period']))
df.head(3)

## 2. Text Cleanup — Mojibake Repair

The export mangles apostrophes and curly quotes into `?` ("don?t", "I?m"). This repairs them
without touching genuine question marks.

It also flags responses hitting the **255-character cap** on the two comment fields — those were
truncated mid-sentence at collection time, so the text you have is incomplete.

In [ ]:
TEXT_COLS = ['do_well', 'do_better', 'barriers_comment',
             'beyond_scope_comment', 'additional_comments']

CONTRACTIONS = r"(?:t|s|re|ve|ll|d|m|T|S|RE|VE|LL|D|M)"

def repair_text(x):
    """Repair '?' that should be an apostrophe.

    Only touches '?' sitting between two word characters (don?t, I?m, y?all).
    A '?' after a space, before punctuation, or at the end of a sentence is left
    alone - those are either real questions or stripped emoji, and rewriting them
    as quote marks produced worse text than leaving them.
    """
    if pd.isna(x):
        return x
    s = str(x)
    s = re.sub(r"(\w)\?(" + CONTRACTIONS + r")\b", r"\1'\2", s)   # don?t -> don't
    s = re.sub(r"(\w)\?(\w)", r"\1'\2", s)                       # y?all -> y'all
    return s.strip()

# Count what changes, then apply
n_repaired = sum(
    (df[c].notna() & (df[c].astype(str) != df[c].apply(repair_text).astype(str))).sum()
    for c in TEXT_COLS
)

for c in TEXT_COLS:
    df[c] = df[c].apply(repair_text)

print(f'Repaired mojibake in {n_repaired} field values')

# Flag truncation at the 255-char collection cap
CAP = 255
for c in ['barriers_comment', 'beyond_scope_comment']:
    flag = c + '_truncated'
    df[flag] = df[c].notna() & (df[c].astype(str).str.len() >= CAP - 3)
    print(f'{c}: {df[flag].sum()} responses truncated at the {CAP}-char cap')

print()
print('Sample of repaired text:')
for t in df.loc[df['do_better'].astype(str).str.contains("'", na=False), 'do_better'].head(3):
    print('  -', t[:90])

## 3. Quantitative Overview — Focus Period

**These charts describe the focus period only** (Jun 2025 – Aug 2026, n=355), so the figures here
can be quoted directly in a focus-period report without silently averaging in 2024 data.

The one exception is the NPS-by-period chart, which is comparative by design.

Section 4 handles every focus-vs-prior comparison.

In [ ]:
# All of section 3 runs on the focus period alone
foc = df[df['period'] == 'focus'].copy()
print(f'Focus period: {len(foc)} responses, '
      f'{foc["date"].min().date()} to {foc["date"].max().date()}')

In [ ]:
# NPS distribution and categories - focus period
nps_score = ((foc['nps_category'] == 'Promoter').mean()
             - (foc['nps_category'] == 'Detractor').mean()) * 100

print(f'Net Promoter Score (focus period): {nps_score:.1f}')
print(foc['nps_category'].value_counts().to_string())
print()
print('For reference, both periods:')
for p, g in df.groupby('period'):
    s = ((g['nps_category'] == 'Promoter').mean()
         - (g['nps_category'] == 'Detractor').mean()) * 100
    print(f'  {p:6s} NPS = {s:5.1f}   (n={len(g)})')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(foc['nps'], bins=10, discrete=True, ax=axes[0],
             color='steelblue', edgecolor='white')
axes[0].set_title('Net Promoter Score Distribution')
axes[0].set_xlabel('Score (0-10)')
axes[0].set_ylabel('Number of clients')
for rng_, color in [(range(0, 7), 'tomato'), (range(7, 9), 'gold'),
                    (range(9, 11), 'mediumseagreen')]:
    for xi in rng_:
        axes[0].axvspan(xi - 0.5, xi + 0.5, alpha=0.08, color=color)

cat_counts = foc['nps_category'].value_counts()
colors = {'Promoter': 'mediumseagreen', 'Passive': 'gold', 'Detractor': 'tomato'}
bars = axes[1].bar(cat_counts.index, cat_counts.values,
                   color=[colors.get(c, 'grey') for c in cat_counts.index])
axes[1].bar_label(bars, fmt='%d', padding=3)
axes[1].set_title(f'NPS Categories  (NPS = {nps_score:.0f})')
axes[1].set_ylabel('Count')

plt.suptitle(f'Net Promoter Score - Focus Period (n={len(foc)})',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_nps_focus.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# NPS by period, side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (p, g) in zip(axes, df.groupby('period', sort=False)):
    sns.histplot(g['nps'], bins=10, discrete=True, ax=ax,
                 color='steelblue', edgecolor='white')
    s = ((g['nps_category'] == 'Promoter').mean()
         - (g['nps_category'] == 'Detractor').mean()) * 100
    ax.set_title(f'{p}  (n={len(g)}, NPS={s:.0f}, mean={g["nps"].mean():.2f})')
    ax.set_xlabel('Score (0-10)')
    ax.set_ylabel('Number of clients')

plt.suptitle('NPS Distribution by Period', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_nps_by_period.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Positive difference.
# Order is derived from the numeric column rather than hardcoded, so a relabelled
# or newly added category can never be silently dropped from the chart.
pos_diff = foc['positive_diff'].value_counts()
order = list(PD_ORDER[::-1])
b = axes[0].bar(order, [pos_diff.get(o, 0) for o in order],
                color=[PD_COLORS[o] for o in order])
axes[0].bar_label(b, fmt='%d', padding=3)
axes[0].set_title('How much of a positive difference\nhas ODL had?')
axes[0].set_ylabel('Count')

# Degree of resolution
res = foc['resolution'].value_counts().dropna()
res_colors = {'Positive': 'mediumseagreen', 'Neutral': 'steelblue', 'Negative': 'tomato'}
b = axes[1].bar(res.index, res.values,
                color=[res_colors.get(r, 'grey') for r in res.index])
axes[1].bar_label(b, fmt='%d', padding=3)
axes[1].set_title('Degree of Resolution')
axes[1].set_ylabel('Count')

plt.suptitle(f'Client Satisfaction Ratings - Focus Period (n={len(foc)})',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_ratings_focus.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Yes/No questions
questions = {
    'Informed about case': foc['informed'],
    'Gave choices':        foc['gave_choices'],
    'Barriers to service': foc['barriers'],
    'Beyond scope':        foc['beyond_scope'],
}

fig, axes = plt.subplots(1, len(questions), figsize=(14, 4))
for ax, (label, col) in zip(axes, questions.items()):
    if pd.api.types.is_numeric_dtype(col):
        # 'Informed about case' is a 0-100 score, not Yes/No - show its shape
        sns.histplot(col.dropna(), bins=20, ax=ax, color='steelblue', edgecolor='white')
        ax.set_title(f'{label}\n(median {col.median():.0f}, {100*(col>=90).mean():.0f}% >= 90)',
                     fontsize=10)
        ax.set_xlabel('Score (0-100)')
    else:
        counts = col.value_counts()
        cols_ = ['mediumseagreen' if i == 'Yes' else 'tomato' if i == 'No' else 'steelblue'
                 for i in counts.index]
        bars = ax.bar(counts.index, counts.values, color=cols_)
        ax.bar_label(bars, fmt='%d', padding=2)
        pct = counts / counts.sum() * 100
        ax.set_title(f'{label}\n({pct.get("Yes", 0):.0f}% Yes)', fontsize=10)
    ax.set_ylabel('Count' if ax is axes[0] else '')

plt.suptitle(f'Response Distributions - Focus Period (n={len(foc)})',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_yesno_focus.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# NPS by case owner, focus period only (owners with 5+ focus-period responses).
# Restricting to one period matters here: pooling would rank staff who were only
# active in 2024 alongside current staff with no way to tell them apart.
owner_counts  = foc['case_owner'].value_counts()
active_owners = owner_counts[owner_counts >= 5].index

owner_nps = (foc[foc['case_owner'].isin(active_owners)]
             .groupby('case_owner')['nps'].agg(['mean', 'count'])
             .round(2).sort_values('mean', ascending=False))

print(f'{len(active_owners)} case owners with >=5 responses in the focus period '
      f'(of {foc["case_owner"].nunique()} total)')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(owner_nps.index, owner_nps['mean'], color='steelblue')
ax.axvline(foc['nps'].mean(), color='tomato', linestyle='--',
           label=f'Focus-period mean ({foc["nps"].mean():.1f})')
for i, (_, row) in enumerate(owner_nps.iterrows()):
    ax.text(row['mean'] + 0.05, i, f"{row['mean']:.1f} (n={int(row['count'])})",
            va='center', fontsize=8)
ax.set_title('Average NPS by Case Owner - Focus Period (>=5 responses)')
ax.set_xlabel('Mean NPS')
ax.set_xlim(0, 11)
ax.legend()
plt.tight_layout()
plt.savefig('fig_nps_by_owner_focus.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Statistical Comparison — Focus vs Prior

Every test below reports a **p-value, an effect size, and a 95% confidence interval**.

A CI that comfortably straddles zero is not merely "not significant" — it tells you how large a
difference you could have detected if one existed. With ~355 responses per period, a narrow CI
around zero is positive evidence of *no meaningful change*, which is a stronger and more useful
statement than "p > 0.05".

In [ ]:
# ── Effect-size helpers ────────────────────────────────────────────────────

def cohens_d(a, b):
    """Standardised mean difference (a - b), pooled SD."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    n1, n2 = len(a), len(b)
    sp = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan


def rank_biserial(a, b):
    """Rank-biserial correlation from Mann-Whitney U. Ranges -1..1, 0 = no shift."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    u = stats.mannwhitneyu(a, b, alternative='two-sided').statistic
    return 2 * u / (len(a) * len(b)) - 1


def hodges_lehmann(a, b):
    """Median of all pairwise differences (a - b). Robust location shift."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.median(np.subtract.outer(a, b).ravel()))


def boot_ci(a, b, fn, n_boot=N_BOOT, alpha=0.05):
    """Percentile bootstrap CI for any two-sample statistic fn(a, b)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    stats_ = np.empty(n_boot)
    for i in range(n_boot):
        stats_[i] = fn(RNG.choice(a, len(a), replace=True),
                       RNG.choice(b, len(b), replace=True))
    lo, hi = np.nanpercentile(stats_, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)


def welch_meandiff_ci(a, b, alpha=0.05):
    """Welch CI for the difference in means (a - b)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    n1, n2 = len(a), len(b)
    v1, v2 = a.var(ddof=1), b.var(ddof=1)
    se = np.sqrt(v1 / n1 + v2 / n2)
    dof = (v1 / n1 + v2 / n2) ** 2 / ((v1 / n1) ** 2 / (n1 - 1) + (v2 / n2) ** 2 / (n2 - 1))
    t = stats.t.ppf(1 - alpha / 2, dof)
    diff = a.mean() - b.mean()
    return diff, (diff - t * se, diff + t * se), dof


RESULTS = []   # collected for export

def record(name, test, stat_label, stat, p, es_label, es, ci):
    RESULTS.append({'Comparison': name, 'Test': test,
                    'Statistic': f'{stat_label}={stat:.3f}', 'p_value': round(p, 4),
                    'Effect size': es_label, 'Estimate': round(es, 3),
                    'CI_low': round(ci[0], 3), 'CI_high': round(ci[1], 3)})

print('Helpers ready.')

### 4.1 Net Promoter Score — focus vs prior (Welch's t-test)

In [ ]:
a = df.loc[df['period'] == 'focus', 'nps'].dropna().values
b = df.loc[df['period'] == 'prior', 'nps'].dropna().values

t, p = stats.ttest_ind(a, b, equal_var=False)
diff, (lo, hi), dof = welch_meandiff_ci(a, b)
d = cohens_d(a, b)
d_lo, d_hi = boot_ci(a, b, cohens_d)

print('Net Promoter Score: focus vs prior')
print(f'  focus  mean = {a.mean():.3f}  (SD {a.std(ddof=1):.3f}, n={len(a)})')
print(f'  prior  mean = {b.mean():.3f}  (SD {b.std(ddof=1):.3f}, n={len(b)})')
print()
print(f"  Welch's t({dof:.1f}) = {t:.3f},  p = {p:.4f}")
print(f'  Mean difference   = {diff:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]')
print(f"  Cohen's d         = {d:+.3f}   95% CI [{d_lo:+.3f}, {d_hi:+.3f}]")

record('NPS: focus vs prior', "Welch's t-test", 't', t, p, "Cohen's d", d, (d_lo, d_hi))

### 4.2 Scale-effect estimate — "Needs met" wording change

The needs-met question changed from a **quality** scale ("Extremely well" … "Not well at all")
to an **agreement** scale ("Strongly agree" … "Strongly disagree") on **4 Nov 2025**, partway
through the focus period.

That timing is useful: within the focus period alone, both wordings are present. Comparing them
*inside a single period* holds time constant, so any gap is attributable to the wording rather
than to a real change in client experience.

The complementary check is the legacy scale compared across periods — same wording, different
time. If that one is flat while the within-period comparison is not, the difference is an
artefact of the instrument.

**Because of this, the two wordings are never pooled into a single mean anywhere in this
notebook.**

In [ ]:
foc = df[df['period'] == 'focus']
cur = foc.loc[foc['needs_scale'] == 'current', 'needs_met_num'].dropna().values
leg = foc.loc[foc['needs_scale'] == 'legacy',  'needs_met_num'].dropna().values

t, p = stats.ttest_ind(cur, leg, equal_var=False)
diff, (lo, hi), dof = welch_meandiff_ci(cur, leg)
d = cohens_d(cur, leg)
d_lo, d_hi = boot_ci(cur, leg, cohens_d)

print('SCALE EFFECT - within the focus period only (time held constant)')
print(f'  current (agreement)  mean = {cur.mean():.3f}  (SD {cur.std(ddof=1):.3f}, n={len(cur)})')
print(f'  legacy  (quality)    mean = {leg.mean():.3f}  (SD {leg.std(ddof=1):.3f}, n={len(leg)})')
print()
print(f"  Welch's t({dof:.1f}) = {t:.3f},  p = {p:.4f}")
print(f'  Mean difference   = {diff:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]')
print(f"  Cohen's d         = {d:+.3f}   95% CI [{d_lo:+.3f}, {d_hi:+.3f}]")
print()
print(f'  Top-box (score 5):  agreement {100*(cur==5).mean():.1f}%  vs  quality {100*(leg==5).mean():.1f}%')
print(f'  Top-2   (score>=4): agreement {100*(cur>=4).mean():.1f}%  vs  quality {100*(leg>=4).mean():.1f}%')

record('Needs met: agreement vs quality wording (within focus)',
       "Welch's t-test", 't', t, p, "Cohen's d", d, (d_lo, d_hi))

# Control: same wording, different period
lf = df[(df['needs_scale'] == 'legacy') & (df['period'] == 'focus')]['needs_met_num'].dropna().values
lp = df[(df['needs_scale'] == 'legacy') & (df['period'] == 'prior')]['needs_met_num'].dropna().values

t2, p2 = stats.ttest_ind(lf, lp, equal_var=False)
diff2, (lo2, hi2), dof2 = welch_meandiff_ci(lf, lp)
d2 = cohens_d(lf, lp)
d2_lo, d2_hi = boot_ci(lf, lp, cohens_d)

print()
print('CONTROL - legacy wording only, focus vs prior (wording held constant)')
print(f'  focus  mean = {lf.mean():.3f}  (n={len(lf)})')
print(f'  prior  mean = {lp.mean():.3f}  (n={len(lp)})')
print(f"  Welch's t({dof2:.1f}) = {t2:.3f},  p = {p2:.4f}")
print(f'  Mean difference   = {diff2:+.3f}   95% CI [{lo2:+.3f}, {hi2:+.3f}]')
print(f"  Cohen's d         = {d2:+.3f}   95% CI [{d2_lo:+.3f}, {d2_hi:+.3f}]")

record('Needs met (legacy wording only): focus vs prior',
       "Welch's t-test", 't', t2, p2, "Cohen's d", d2, (d2_lo, d2_hi))

In [ ]:
# Visualise the scale effect
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: response distribution under each wording, within focus
w = 0.38
xs = np.arange(1, 6)
cur_pct = [100 * (cur == v).mean() for v in xs]
leg_pct = [100 * (leg == v).mean() for v in xs]
axes[0].bar(xs - w/2, leg_pct, w, label=f'quality wording (n={len(leg)})', color='steelblue')
axes[0].bar(xs + w/2, cur_pct, w, label=f'agreement wording (n={len(cur)})', color='mediumseagreen')
axes[0].set_xticks(xs)
axes[0].set_xlabel('Response (1 = worst, 5 = best)')
axes[0].set_ylabel('% of responses')
axes[0].set_title('Same period, different wording')
axes[0].legend(fontsize=8)

# Right: means with 95% CIs for all three groups
groups = [('prior\nquality', lp), ('focus\nquality', lf), ('focus\nagreement', cur)]
means = [g.mean() for _, g in groups]
errs  = [1.96 * g.std(ddof=1) / np.sqrt(len(g)) for _, g in groups]
cols_ = ['steelblue', 'steelblue', 'mediumseagreen']
axes[1].bar([g[0] for g in groups], means, yerr=errs, capsize=5, color=cols_)
for i, (m, e) in enumerate(zip(means, errs)):
    axes[1].text(i, m + e + 0.05, f'{m:.2f}', ha='center', fontsize=9)
axes[1].set_ylabel('Mean needs-met score (1-5)')
axes[1].set_ylim(0, 5.6)
axes[1].set_title('The jump tracks the wording, not the period')

plt.suptitle('Needs Met - Scale Change Artefact', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_scale_effect.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Positive difference — focus vs prior (Mann-Whitney U)

In [ ]:
a = df.loc[df['period'] == 'focus', 'positive_diff_num'].dropna().values
b = df.loc[df['period'] == 'prior', 'positive_diff_num'].dropna().values

u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
r = rank_biserial(a, b)
r_lo, r_hi = boot_ci(a, b, rank_biserial)
hl = hodges_lehmann(a, b)
hl_lo, hl_hi = boot_ci(a, b, hodges_lehmann)

print('Positive difference ODL has made: focus vs prior  (ordinal -> Mann-Whitney U)')
print(f'  focus  median = {np.median(a):.1f}  mean = {a.mean():.3f}  (n={len(a)})')
print(f'  prior  median = {np.median(b):.1f}  mean = {b.mean():.3f}  (n={len(b)})')
print()
print(f'  Mann-Whitney U = {u:.1f},  p = {p:.4f}')
print(f'  Rank-biserial r      = {r:+.3f}   95% CI [{r_lo:+.3f}, {r_hi:+.3f}]')
print(f'  Hodges-Lehmann shift = {hl:+.3f}   95% CI [{hl_lo:+.3f}, {hl_hi:+.3f}]')

record('Positive difference: focus vs prior', 'Mann-Whitney U', 'U', u, p,
       'Rank-biserial r', r, (r_lo, r_hi))

In [ ]:
# Stacked bars: composition of responses in each period
ct = (pd.crosstab(df['period'], df['positive_diff'], normalize='index') * 100)
ct = ct[PD_ORDER].reindex(['prior', 'focus'])

# Every category must be represented or the bars will not reach 100%
assert set(ct.columns) == set(df['positive_diff'].dropna().unique()), 'category dropped'
assert np.allclose(ct.sum(axis=1), 100), f'bars sum to {ct.sum(axis=1).values}, not 100'

fig, ax = plt.subplots(figsize=(11, 3.4))
left = np.zeros(len(ct))
for cat in ct.columns:
    vals = ct[cat].values
    ax.barh(ct.index, vals, left=left, color=PD_COLORS[cat],
            edgecolor='white', label=cat)
    for i, (v, l) in enumerate(zip(vals, left)):
        if v >= 4:
            ax.text(l + v/2, i, f'{v:.0f}%', ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
    left += vals

ax.set_xlim(0, 100)
ax.set_xlabel('% of responses')
ax.set_title(f'"How much of a positive difference has ODL had?"   '
             f'(Mann-Whitney p = {p:.3f}, rank-biserial r = {r:+.3f})')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.grid(False)
plt.tight_layout()
plt.savefig('fig_positive_diff_stacked.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Gave choices and barriers reported — focus vs prior (two-proportion z-tests)

In [ ]:
def prop_test(col, label):
    a = df.loc[df['period'] == 'focus', col].dropna()
    b = df.loc[df['period'] == 'prior', col].dropna()
    s1, n1 = int(a.sum()), len(a)
    s2, n2 = int(b.sum()), len(b)
    p1, p2 = s1 / n1, s2 / n2

    z, p = proportions_ztest([s1, s2], [n1, n2])
    lo, hi = confint_proportions_2indep(s1, n1, s2, n2,
                                        compare='diff', method='newcomb')

    print(f'{label}: focus vs prior')
    print(f'  focus  {p1*100:5.1f}%  ({s1}/{n1})')
    print(f'  prior  {p2*100:5.1f}%  ({s2}/{n2})')
    print(f'  z = {z:.3f},  p = {p:.4f}')
    print(f'  Risk difference = {(p1-p2)*100:+.1f} pp   95% CI [{lo*100:+.1f}, {hi*100:+.1f}] pp')
    print()

    record(f'{label}: focus vs prior', 'Two-proportion z-test', 'z', z, p,
           'Risk difference (pp)', (p1 - p2) * 100, (lo * 100, hi * 100))
    return p1, p2, (lo, hi), p

gc = prop_test('gave_choices_bin',  'Gave choices')
br = prop_test('barriers_bin',      'Barriers reported')

In [ ]:
# Proportions with 95% CIs on the difference
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (res, label, good_up) in zip(axes, [(gc, 'Gave choices', True),
                                            (br, 'Barriers reported', False)]):
    p1, p2, (lo, hi), pv = res
    bars = ax.bar(['prior', 'focus'], [p2 * 100, p1 * 100],
                  color=['lightsteelblue', 'steelblue'])
    ax.bar_label(bars, fmt='%.1f%%', padding=3)
    ax.set_ylabel('% Yes')
    ax.set_ylim(0, max(p1, p2) * 100 * 1.35)
    ax.set_title(f'{label}\n'
                 f'diff {(p1-p2)*100:+.1f} pp, 95% CI [{lo*100:+.1f}, {hi*100:+.1f}], p={pv:.3f}',
                 fontsize=10)

plt.suptitle('Binary Measures - Focus vs Prior', fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('fig_proportions.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Informed about case — focus vs prior (Mann-Whitney U)

This variable is a 0-100 score that piles up at the ceiling — 46% of responses are exactly 100,
with a separate spike at 0. The mean (81.3) describes almost nobody, so the comparison uses the
**median** and a rank-based test instead, and the distribution is plotted so the cluster at 0
stays visible.

In [ ]:
a = df.loc[df['period'] == 'focus', 'informed'].dropna().values
b = df.loc[df['period'] == 'prior', 'informed'].dropna().values

u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
r = rank_biserial(a, b)
r_lo, r_hi = boot_ci(a, b, rank_biserial)
hl = hodges_lehmann(a, b)
hl_lo, hl_hi = boot_ci(a, b, hodges_lehmann)

print('Informed about case: focus vs prior')
print(f'  focus  median = {np.median(a):.1f}   %>=90 = {100*(a>=90).mean():.1f}%   '
      f'%==0 = {100*(a==0).mean():.1f}%   (n={len(a)})')
print(f'  prior  median = {np.median(b):.1f}   %>=90 = {100*(b>=90).mean():.1f}%   '
      f'%==0 = {100*(b==0).mean():.1f}%   (n={len(b)})')
print()
print(f'  Mann-Whitney U = {u:.1f},  p = {p:.4f}')
print(f'  Rank-biserial r      = {r:+.3f}   95% CI [{r_lo:+.3f}, {r_hi:+.3f}]')
print(f'  Hodges-Lehmann shift = {hl:+.3f}   95% CI [{hl_lo:+.3f}, {hl_hi:+.3f}]')

record('Informed about case: focus vs prior', 'Mann-Whitney U', 'U', u, p,
       'Rank-biserial r', r, (r_lo, r_hi))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Distribution, both periods overlaid
bins = np.arange(0, 105, 5)
axes[0].hist([b, a], bins=bins, label=['prior', 'focus'],
             color=['lightsteelblue', 'steelblue'])
axes[0].set_xlabel('Informed about case (0-100)')
axes[0].set_ylabel('Number of clients')
axes[0].set_title('Ceiling-piled, with a separate cluster at 0')
axes[0].legend()

# Medians and threshold rates
labels = ['median', '% >= 90', '% == 0']
prior_v = [np.median(b), 100 * (b >= 90).mean(), 100 * (b == 0).mean()]
focus_v = [np.median(a), 100 * (a >= 90).mean(), 100 * (a == 0).mean()]
x, w = np.arange(len(labels)), 0.38
axes[1].bar(x - w/2, prior_v, w, label='prior', color='lightsteelblue')
axes[1].bar(x + w/2, focus_v, w, label='focus', color='steelblue')
for i, (pv, fv) in enumerate(zip(prior_v, focus_v)):
    axes[1].text(i - w/2, pv + 1, f'{pv:.0f}', ha='center', fontsize=9)
    axes[1].text(i + w/2, fv + 1, f'{fv:.0f}', ha='center', fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_title(f'Robust summaries  (Mann-Whitney p = {p:.3f})')
axes[1].legend()

plt.suptitle('Informed About Case', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_informed.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.6 Informed about case — monthly trend

The focus-vs-prior test above buckets 29 months into two blocks and finds no difference. A monthly
view shows movement the two-bucket split hides. Both panels track the same variable: the left is
the **share of clients who felt well-informed** (score ≥ 90), the right is the **share who felt
completely uninformed** (score 0) — the detractor spike that the mean buries.

Marker area is proportional to that month's response count, so thin months read as less certain;
months with fewer than 8 responses are drawn hollow and left off the trend line, and the final
month (Aug 2026) is partial.

In [ ]:
import matplotlib.dates as mdates

MIN_N_MONTH = 8

gm = (df.dropna(subset=['informed', 'date'])
        .set_index('date').resample('ME')['informed']
        .agg(n='count',
             p90=lambda s: 100 * (s >= 90).mean(),
             p0=lambda s: 100 * (s == 0).mean()))
gm_full = gm[gm.n >= MIN_N_MONTH]
gm_thin = gm[gm.n < MIN_N_MONTH]
FOCUS_START = pd.Timestamp('2025-06-01')


def informed_trend(themed, fname):
    if themed:
        surface, ink, muted, grid = SLIDE['surface'], SLIDE['ink'], SLIDE['muted'], SLIDE['grid']
        c90, c0, shade = SLIDE['focus'], '#E2534C', '#0A1040'
    else:
        surface, ink, muted, grid = 'white', 'black', 'dimgray', '#d9d9d9'
        c90, c0, shade = 'steelblue', '#c0392b', '#eef2fb'

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), facecolor=surface)
    panels = [(axes[0], 'p90', '% of clients scoring >= 90', c90, 'Well-informed clients'),
              (axes[1], 'p0',  '% of clients scoring 0', c0, 'Felt uninformed (score 0)')]
    for ax, col, ylab, color, title in panels:
        ax.set_facecolor(surface)
        ax.axvspan(FOCUS_START, gm.index.max(), color=shade, zorder=0)
        ax.plot(gm_full.index, gm_full[col], color=color, lw=2, zorder=3)
        ax.scatter(gm_full.index, gm_full[col], s=np.clip(gm_full.n * 2.2, 20, 260),
                   color=color, edgecolor=surface, linewidth=1.2, zorder=4)
        if len(gm_thin):
            ax.scatter(gm_thin.index, gm_thin[col], s=40, facecolor='none',
                       edgecolor=muted, linewidth=1.2, zorder=4)
        overall = 100 * ((df['informed'] >= 90).mean() if col == 'p90'
                         else (df['informed'] == 0).mean())
        ax.axhline(overall, color=muted, ls='--', lw=1, zorder=2, label=f'Overall {overall:.0f}%')
        ymax = max(gm_full[col].max() * 1.15, overall * 1.3)
        ax.text(FOCUS_START + (gm.index.max() - FOCUS_START) / 2, ymax * 0.96, 'focus period',
                ha='center', va='top', color=muted, fontsize=9, style='italic')
        ax.set_ylim(0, ymax)
        ax.set_ylabel(ylab, color=ink)
        ax.set_title(title, color=ink, fontsize=12, pad=8)
        ax.legend(frameon=False, labelcolor=ink, loc='upper right', fontsize=9)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
        ax.tick_params(colors=muted)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        for s in ('top', 'right'):
            ax.spines[s].set_visible(False)
        for s in ('left', 'bottom'):
            ax.spines[s].set_color(grid)
        ax.set_axisbelow(True)
        ax.grid(True, axis='y', color=grid, lw=0.8)
        for gl in ax.get_ygridlines():
            gl.set_color(grid)

    fig.suptitle('Informed About Case - Monthly Trend', color=ink,
                 fontweight='bold', fontsize=15, y=1.02)
    fig.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches='tight', facecolor=surface)
    plt.show()
    print(f'  saved {fname}')


informed_trend(False, 'fig_informed_trend.png')
informed_trend(True,  'fig_informed_trend_themed.png')

### 4.7 All results together

In [ ]:
results_df = pd.DataFrame(RESULTS)
results_df['95% CI'] = results_df.apply(
    lambda r: f"[{r['CI_low']:+.3f}, {r['CI_high']:+.3f}]", axis=1)

display(results_df[['Comparison', 'Test', 'Statistic', 'p_value',
                    'Effect size', 'Estimate', '95% CI']])

results_df.to_csv('statistical_results.csv', index=False)
print('\nSaved -> statistical_results.csv')

## 5. Translate Responses to English

Topic modeling counts word co-occurrence, so untranslated multilingual text splits into
language-based clusters instead of theme-based ones. Translating first removes that.

The focus period is 305 English and 48 Spanish responses; the prior period also runs through the
same step. Unique strings are translated once and reused, which cuts the API calls by roughly
half since many answers repeat ("Everything", "Nothing", "idk").

In [ ]:
def translate_series(s: pd.Series) -> pd.Series:
    """Translate a text column to English, caching each unique string."""
    SKIP = {'', 'nan', 'n/a', 'none', 'no', 'na', '?'}
    texts = s.dropna().astype(str).str.strip()
    uniques = sorted({t for t in texts if t.lower() not in SKIP})

    cache = {}
    for i, t in enumerate(uniques, 1):
        try:
            cache[t] = GoogleTranslator(source='auto', target='en').translate(t) or t
        except Exception:
            cache[t] = t
        if i % 50 == 0:
            print(f'    {i}/{len(uniques)} unique strings')

    def apply_one(x):
        if pd.isna(x):
            return ''
        t = str(x).strip()
        return '' if t.lower() in SKIP else cache.get(t, t)

    return s.apply(apply_one)


for col in ['do_well', 'do_better']:
    print(f'Translating "{col}" ...')
    df[col + '_en'] = translate_series(df[col])
    print(f'  done ({(df[col + "_en"].str.len() > 0).sum()} non-empty)')

print()
print('Spot-check:')
mask = df['do_well'].notna() & (df['do_well'].astype(str) != df['do_well_en'])
display(df.loc[mask, ['do_well', 'do_well_en']].head(5))

In [ ]:
def clean_text(text) -> str:
    if pd.isna(text) or str(text).strip() == '':
        return ''
    t = str(text).lower()
    t = re.sub(r'http\S+|www\.\S+', '', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return ' '.join(w for w in t.split() if w not in ALL_STOPS and len(w) > 2)


df['do_well_clean']   = df['do_well_en'].apply(clean_text)
df['do_better_clean'] = df['do_better_en'].apply(clean_text)

dw = df[df['do_well_clean'].str.len() > 0].copy()
db = df[df['do_better_clean'].str.len() > 0].copy()

print('Responses with usable text after cleaning:')
print(f'  do_well   {len(dw):4d}   ' + '  '.join(
    f'{p}={n}' for p, n in dw['period'].value_counts().items()))
print(f'  do_better {len(db):4d}   ' + '  '.join(
    f'{p}={n}' for p, n in db['period'].value_counts().items()))

### Slide theme

The topic charts are the only figures this notebook produces that `make_themed_figures.py`
cannot, because they depend on the translation step. So each one is rendered twice — once in
the default light styling and once in the slide deck's dark-navy palette — and both are pushed.
No toggle to set; you get both files either way.

Palette and rationale are documented in `make_themed_figures.py`.

In [ ]:
def plot_topic_prevalence(dfx, title, fname, themed):
    """Topic prevalence by period. Renders light or slide-themed from one code path."""
    ct = pd.crosstab(dfx['topic_label'], dfx['period'], normalize='columns') * 100
    ct = ct.reindex(columns=['prior', 'focus'])
    counts = pd.crosstab(dfx['topic_label'], dfx['period']).reindex(columns=['prior', 'focus'])
    ct = ct.loc[ct.mean(axis=1).sort_values().index]

    if themed:
        c_prior, c_focus = SLIDE['prior'], SLIDE['focus']
        ink, muted, surface, grid = (SLIDE['ink'], SLIDE['muted'],
                                     SLIDE['surface'], SLIDE['grid'])
    else:
        c_prior, c_focus = 'lightsteelblue', 'steelblue'
        ink, muted, surface, grid = 'black', 'dimgray', 'white', '#d0d0d0'

    fig, ax = plt.subplots(figsize=(11, 5), facecolor=surface)
    ax.set_facecolor(surface)
    y, h = np.arange(len(ct)), 0.38
    # edgecolor is set explicitly: the seaborn whitegrid style applied in the setup
    # cell draws white patch edges, which halo every bar against a dark surface.
    ax.barh(y - h/2, ct['prior'], h, label='prior', color=c_prior, edgecolor=surface)
    ax.barh(y + h/2, ct['focus'], h, label='focus', color=c_focus, edgecolor=surface)
    for i, idx in enumerate(ct.index):
        ax.text(ct.loc[idx, 'prior'] + 0.4, i - h/2,
                f"{ct.loc[idx,'prior']:.0f}% (n={counts.loc[idx,'prior']})",
                va='center', fontsize=8, color=muted)
        ax.text(ct.loc[idx, 'focus'] + 0.4, i + h/2,
                f"{ct.loc[idx,'focus']:.0f}% (n={counts.loc[idx,'focus']})",
                va='center', fontsize=8, color=muted)

    ax.set_yticks(y)
    ax.set_yticklabels(ct.index, color=ink)
    ax.set_xlabel('% of responses within period', color=ink)
    ax.set_xlim(0, max(ct.max()) * 1.28)
    ax.set_title(title, color=ink, fontweight='bold', fontsize=13, pad=12)
    ax.tick_params(colors=muted)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(grid)
    ax.set_axisbelow(True)
    ax.grid(False)
    ax.grid(True, axis='x', color=grid, linewidth=0.8)
    for gl in ax.get_xgridlines():
        gl.set_color(grid)
    # Outside the axes: the value labels occupy the right edge of the plot area, so an
    # inside legend lands on the shortest topic's bars.
    leg = ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1))
    for t in leg.get_texts():
        t.set_color(ink)

    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=surface)
    plt.show()
    print(f'  saved {fname}')

## 6. Topic Modeling — What ODL Does Well

**How the periods are compared.** A single LDA model is fitted on the *combined* focus and prior
text, then each period's share of every topic is measured. Fitting one model per period would
produce two independent sets of topics — "Topic 2" in one would have no relationship to "Topic 2"
in the other, so the two could not be placed side by side. One shared topic space keeps the
comparison meaningful.

Run the cell below, read the top words, then fill in `TOPIC_LABELS_WELL` in the cell after it.

In [ ]:
N_TOPICS_WELL = 5
N_TOP_WORDS   = 8

vec_dw  = CountVectorizer(max_features=3000, min_df=2, ngram_range=(1, 2))
mat_dw  = vec_dw.fit_transform(dw['do_well_clean'])
vocab_dw = vec_dw.get_feature_names_out()

lda_dw = LatentDirichletAllocation(n_components=N_TOPICS_WELL, random_state=42,
                                   learning_method='batch', max_iter=30)
lda_dw.fit(mat_dw)

doc_topics_dw = lda_dw.transform(mat_dw)
dw['topic']        = doc_topics_dw.argmax(axis=1)
dw['topic_weight'] = doc_topics_dw.max(axis=1)


def top_words_table(model, vocab, n=8):
    return pd.DataFrame([
        {'Topic': f'Topic {i}',
         'Top Words': ', '.join(vocab[j] for j in comp.argsort()[:-n - 1:-1])}
        for i, comp in enumerate(model.components_)
    ])


print('Topics - What ODL does well  (fitted on both periods together):')
display(top_words_table(lda_dw, vocab_dw, N_TOP_WORDS))
print('Documents per topic by period:')
display(pd.crosstab(dw['topic'], dw['period']))

In [ ]:
# ── Label topics after reading the top words above ────────────────────────
TOPIC_LABELS_WELL = {
    0: 'Topic 0',
    1: 'Topic 1',
    2: 'Topic 2',
    3: 'Topic 3',
    4: 'Topic 4',
}
dw['topic_label'] = dw['topic'].map(TOPIC_LABELS_WELL)
print(dw['topic_label'].value_counts().to_string())

In [ ]:
plot_topic_prevalence(dw, 'What ODL Does Well — Topic Prevalence by Period',
                       'fig_topics_well.png',  themed=False)
plot_topic_prevalence(dw, 'What ODL Does Well — Topic Prevalence by Period',
                       'fig_topics_well_themed.png', themed=True)

## 7. Topic Modeling — What Could Be Better

Same approach: one model fitted across both periods, prevalence compared within each.

In [ ]:
N_TOPICS_BETTER = 5

vec_db  = CountVectorizer(max_features=3000, min_df=2, ngram_range=(1, 2))
mat_db  = vec_db.fit_transform(db['do_better_clean'])
vocab_db = vec_db.get_feature_names_out()

lda_db = LatentDirichletAllocation(n_components=N_TOPICS_BETTER, random_state=42,
                                   learning_method='batch', max_iter=30)
lda_db.fit(mat_db)

doc_topics_db = lda_db.transform(mat_db)
db['topic']        = doc_topics_db.argmax(axis=1)
db['topic_weight'] = doc_topics_db.max(axis=1)

print('Topics - What could be better  (fitted on both periods together):')
display(top_words_table(lda_db, vocab_db, N_TOP_WORDS))
print('Documents per topic by period:')
display(pd.crosstab(db['topic'], db['period']))

In [ ]:
# ── Label topics after reading the top words above ────────────────────────
TOPIC_LABELS_BETTER = {
    0: 'Topic 0',
    1: 'Topic 1',
    2: 'Topic 2',
    3: 'Topic 3',
    4: 'Topic 4',
}
db['topic_label'] = db['topic'].map(TOPIC_LABELS_BETTER)
print(db['topic_label'].value_counts().to_string())

In [ ]:
plot_topic_prevalence(db, 'What Could Be Better — Topic Prevalence by Period',
                       'fig_topics_better.png',  themed=False)
plot_topic_prevalence(db, 'What Could Be Better — Topic Prevalence by Period',
                       'fig_topics_better_themed.png', themed=True)

## 8. Topic Modeling — Combined Issue Comments

Sections 6 and 7 model `do_well` and `do_better`, where the median answer is 7–9 words and a
large share are variants of "everything" or "idk". There is little for a topic model to work
with there.

This section pools the three follow-up comment fields instead:

| Field | Non-empty | Median words |
|---|---|---|
| `barriers_comment` — barriers to service | 111 | 12 |
| `beyond_scope_comment` — anything we could not help with | 198 | 15 |
| `additional_comments` | 3 | 37 |

That is **312 documents**, and they are substantially longer than the `do_well` / `do_better`
answers because clients only fill them in when something specific went wrong. Roughly 120 run to
20 words or more.

Two modelling choices worth stating:

* **Each comment is its own document**, not one document per respondent. A client who reports a
  barrier *and* an unmet need has written two separate thoughts, and concatenating them would
  blur two topics into one. The source field is retained so topics can be traced back to the
  question that produced them.
* **Fitted across both periods at once**, as in sections 6 and 7, so prevalence can be compared
  between them.

A caveat carried over from the earlier sections: 312 documents at ~13 median words is still a
thin corpus for LDA. If the topics below also fail to cohere, the limit is the data rather than
the settings, and a keyword-driven coded-category pass would be the better tool.

In [ ]:
ISSUE_FIELDS = {
    'barriers_comment':     'Barriers to service',
    'beyond_scope_comment': 'Could not help with',
    'additional_comments':  'Additional comments',
}

SKIP_VALUES = {'', 'nan', 'n/a', 'no', 'none', 'na', 'yes', '?', '0',
               'good', 'everything', 'all', 'idk'}

# Long form: one row per comment, tagged with the field it came from
frames = []
for col, label in ISSUE_FIELDS.items():
    sub = df[[col, 'period', 'client_name', 'date']].copy()
    sub['text'] = sub[col].astype(str).str.strip()
    sub = sub[sub[col].notna() & ~sub['text'].str.lower().isin(SKIP_VALUES)]
    sub['source'] = label
    frames.append(sub[['text', 'source', 'period', 'client_name', 'date']])

issues = pd.concat(frames, ignore_index=True)


def drop_cross_field_duplicates(frame):
    """Remove a comment that is a truncated copy of the same client's longer one.

    The two comment fields are capped at 255 characters while Additional Comments
    runs to 311, so an interviewer who filled in both leaves the same answer stored
    twice - once complete, once cut off mid-sentence. Modelling both would weight
    that client's text double. Exact-match dropping misses it because the strings
    differ, so the test is whether the shorter normalises to a prefix of the longer.
    The longest surviving version of each is kept.
    """
    norm = lambda t: re.sub(r'\W+', ' ', str(t)).strip().lower()
    drop = set()
    for _, grp in frame.groupby('client_name'):
        rows = sorted(((i, norm(t)) for i, t in grp['text'].items()),
                      key=lambda kv: len(kv[1]), reverse=True)
        for j, (i_long, long_t) in enumerate(rows):
            if i_long in drop:
                continue
            for i_short, short_t in rows[j + 1:]:
                if len(short_t) >= 25 and long_t.startswith(short_t[:min(len(short_t), 120)]):
                    drop.add(i_short)
    if drop:
        print(f'Dropped {len(drop)} comment(s) duplicated across fields by truncation')
    return frame.drop(index=drop)


issues = drop_cross_field_duplicates(issues).reset_index(drop=True)

print(f'{len(issues)} comments pooled from {len(ISSUE_FIELDS)} fields')
print(issues.groupby(['source', 'period']).size().unstack(fill_value=0).to_string())
print()
print(f'Median words: {issues["text"].str.split().str.len().median():.0f}')
print(f'20+ words:    {(issues["text"].str.split().str.len() >= 20).sum()}')

In [ ]:
print('Translating pooled comments ...')
issues['text_en'] = translate_series(issues['text'])
issues['clean']   = issues['text_en'].apply(clean_text)

issues = issues[issues['clean'].str.len() > 0].copy()
print(f'{len(issues)} comments with usable text after translation and cleaning')
print(issues['period'].value_counts().to_string())

In [ ]:
N_TOPICS_ISSUES = 5

vec_is  = CountVectorizer(max_features=3000, min_df=2, ngram_range=(1, 2))
mat_is  = vec_is.fit_transform(issues['clean'])
vocab_is = vec_is.get_feature_names_out()

lda_is = LatentDirichletAllocation(n_components=N_TOPICS_ISSUES, random_state=42,
                                   learning_method='batch', max_iter=30)
lda_is.fit(mat_is)

doc_topics_is = lda_is.transform(mat_is)
issues['topic']        = doc_topics_is.argmax(axis=1)
issues['topic_weight'] = doc_topics_is.max(axis=1)

print('Topics - combined issue comments:')
display(top_words_table(lda_is, vocab_is, N_TOP_WORDS))
print()
print('Which field feeds each topic:')
display(pd.crosstab(issues['topic'], issues['source']))
print()
print('By period:')
display(pd.crosstab(issues['topic'], issues['period']))

In [ ]:
# Read a few of the most representative comments per topic before naming them -
# the top-words list alone is often ambiguous.
for t in range(N_TOPICS_ISSUES):
    top = (issues[issues['topic'] == t]
           .nlargest(3, 'topic_weight')[['source', 'text_en']])
    print(f'--- Topic {t}  (n={(issues["topic"] == t).sum()}) ---')
    for _, r in top.iterrows():
        print(f'  [{r["source"]}] {r["text_en"][:160]}')
    print()

In [ ]:
# ── Label topics after reading the top words and examples above ───────────
TOPIC_LABELS_ISSUES = {
    0: 'Topic 0',
    1: 'Topic 1',
    2: 'Topic 2',
    3: 'Topic 3',
    4: 'Topic 4',
}
issues['topic_label'] = issues['topic'].map(TOPIC_LABELS_ISSUES)
print(issues['topic_label'].value_counts().to_string())

In [ ]:
plot_topic_prevalence(issues, 'Issue Comments - Topic Prevalence by Period',
                      'fig_topics_issues.png',        themed=False)
plot_topic_prevalence(issues, 'Issue Comments - Topic Prevalence by Period',
                      'fig_topics_issues_themed.png', themed=True)

In [ ]:
def plot_topic_by_source(dfx, fname, themed):
    """Which of the three fields each topic draws from."""
    ct = pd.crosstab(dfx['topic_label'], dfx['source'])
    ct = ct.loc[ct.sum(axis=1).sort_values().index]

    if themed:
        ink, muted, surface, grid = (SLIDE['ink'], SLIDE['muted'],
                                     SLIDE['surface'], SLIDE['grid'])
        colors = ['#4A85E8', '#21A896', '#E0A83A']
    else:
        ink, muted, surface, grid = 'black', 'dimgray', 'white', '#d0d0d0'
        colors = ['steelblue', 'mediumseagreen', 'goldenrod']

    fig, ax = plt.subplots(figsize=(11, 5), facecolor=surface)
    ax.set_facecolor(surface)
    left = np.zeros(len(ct))
    for col, color in zip(ct.columns, colors):
        vals = ct[col].values
        ax.barh(ct.index, vals, left=left, color=color,
                edgecolor=surface, linewidth=2, label=col)
        left += vals
    for i, total in enumerate(ct.sum(axis=1).values):
        ax.text(total + 1, i, str(total), va='center', fontsize=9, color=muted)

    ax.set_xlabel('Number of comments', color=ink)
    ax.set_title('Issue Comments - Which Field Each Topic Draws From',
                 color=ink, fontweight='bold', fontsize=13, pad=12)
    ax.tick_params(colors=muted)
    ax.set_yticklabels(ct.index, color=ink)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(grid)
    ax.set_axisbelow(True)
    ax.grid(False)
    ax.grid(True, axis='x', color=grid, linewidth=0.8)
    for gl in ax.get_xgridlines():
        gl.set_color(grid)
    leg = ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1))
    for t in leg.get_texts():
        t.set_color(ink)

    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=surface)
    plt.show()
    print(f'  saved {fname}')


plot_topic_by_source(issues, 'fig_topics_issues_source.png',        themed=False)
plot_topic_by_source(issues, 'fig_topics_issues_source_themed.png', themed=True)

## 9. Export Results & Push to GitHub

In [ ]:
export_cols = [
    'date', 'period', 'client_name', 'case_number', 'case_owner', 'resolution',
    'nps', 'nps_category', 'positive_diff', 'positive_diff_num',
    'needs_met_num', 'needs_met_source', 'needs_scale',
    'do_well', 'do_well_en', 'do_better', 'do_better_en',
    'informed', 'gave_choices', 'barriers', 'beyond_scope',
    'barriers_comment', 'beyond_scope_comment',
]
export_cols = [c for c in export_cols if c in df.columns]
df[export_cols].to_csv('odl_feedback_results.csv', index=False)
print('Saved -> odl_feedback_results.csv')

top_words_table(lda_dw, vocab_dw, N_TOP_WORDS).to_csv('topics_do_well.csv', index=False)
top_words_table(lda_db, vocab_db, N_TOP_WORDS).to_csv('topics_do_better.csv', index=False)
top_words_table(lda_is, vocab_is, N_TOP_WORDS).to_csv('topics_issues.csv', index=False)
print('Saved -> topics_do_well.csv')
print('Saved -> topics_do_better.csv')
print('Saved -> topics_issues.csv')

# Every pooled comment with its source field, translation and assigned topic
issues[['date', 'period', 'client_name', 'source', 'text', 'text_en',
        'topic', 'topic_label', 'topic_weight']].to_csv('issue_comments_topics.csv', index=False)
print('Saved -> issue_comments_topics.csv')


def format_for_claude(col_en, label, period=None):
    sub = df if period is None else df[df['period'] == period]
    responses = sub[col_en].dropna().astype(str)
    responses = responses[responses.str.strip().str.len() > 0]
    tag = f' - {period}' if period else ''
    lines = [f'## {label}{tag} ({len(responses)} responses)\n']
    lines += [f'{i}. {t.strip()}' for i, t in enumerate(responses, 1)]
    return '\n'.join(lines)


with open('comments_do_well.txt', 'w') as f:
    f.write(format_for_claude('do_well_en', 'What ODL Does Well', 'focus'))
with open('comments_do_better.txt', 'w') as f:
    f.write(format_for_claude('do_better_en', 'What Could Be Better', 'focus'))
print('Saved -> comments_do_well.txt')
print('Saved -> comments_do_better.txt')

with open('comments_issues.txt', 'w') as f:
    sub = issues[issues['period'] == 'focus']
    f.write(f'## Issue comments - focus period ({len(sub)} comments)

')
    for i, (_, r) in enumerate(sub.iterrows(), 1):
        f.write(f'{i}. [{r["source"]}] {r["text_en"].strip()}
')
print('Saved -> comments_issues.txt')

In [ ]:
!pip install PyGithub -q

from github import Github
from google.colab import userdata
from pathlib import Path

# ── Config ─────────────────────────────────────────────────────────────────
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')   # set in Colab's key icon sidebar
REPO_NAME    = 'RyAlah/ODL_Analysis'
BRANCH       = 'claude/sentiment-analysis-notebook-cn8ymo'
OUTPUT_DIR   = 'outputs'
# ───────────────────────────────────────────────────────────────────────────

files_to_push = [
    'odl_feedback_results.csv',
    'statistical_results.csv',
    'topics_do_well.csv',
    'topics_do_better.csv',
    'topics_issues.csv',
    'issue_comments_topics.csv',
    'comments_do_well.txt',
    'comments_do_better.txt',
    'comments_issues.txt',
    # Section 3 - focus period only. The pooled-across-both-periods versions
    # (fig_nps.png, fig_ratings.png, fig_yesno.png, fig_nps_by_owner.png) are
    # kept in outputs/ from an earlier run and are deliberately not regenerated,
    # so they are absent here rather than being overwritten.
    'fig_nps_focus.png',
    'fig_ratings_focus.png',
    'fig_yesno_focus.png',
    'fig_nps_by_owner_focus.png',
    # Section 3 comparative + section 4
    'fig_nps_by_period.png',
    'fig_scale_effect.png',
    'fig_positive_diff_stacked.png',
    'fig_proportions.png',
    'fig_informed.png',
    'fig_informed_trend.png',
    'fig_informed_trend_themed.png',
    'fig_topics_well.png',
    'fig_topics_better.png',
    'fig_topics_well_themed.png',
    'fig_topics_better_themed.png',
    'fig_topics_issues.png',
    'fig_topics_issues_themed.png',
    'fig_topics_issues_source.png',
    'fig_topics_issues_source_themed.png',
]

g    = Github(GITHUB_TOKEN)
repo = g.get_repo(REPO_NAME)

for fname in files_to_push:
    p = Path(fname)
    if not p.exists():
        print(f'Skipped (not found) -> {fname}')
        continue
    content = p.read_bytes()
    path    = f'{OUTPUT_DIR}/{fname}'
    try:
        existing = repo.get_contents(path, ref=BRANCH)
        repo.update_file(path, f'Update {fname}', content, existing.sha, branch=BRANCH)
        print(f'Updated -> {path}')
    except Exception:
        repo.create_file(path, f'Add {fname}', content, branch=BRANCH)
        print(f'Created -> {path}')